In [7]:
import os 
import certifi
import requests 
from dotenv import load_dotenv 

from langchain_openai import ChatOpenAI 
from langchain.tools import tool 
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
from langsmith import Client 


In [64]:
from langchain_classic.agents import create_react_agent, AgentExecutor 

In [56]:
# Load environment variables from .env file
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")


In [ ]:
search_tool = TavilySearchResults(num_results=2)

In [14]:
result = search_tool.invoke("What is lastest AI news?")
result 

[{'title': 'The Latest AI News and Breakthroughs That Matter Most | News',
  'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates',
  'content': "Date: May 21, 2026\n\nSummary: SoundHound AI announced the acquisition of conversational AI platform LivePerson, combining its voice and audio recognition capabilities with LivePerson's digital messaging products. Management has set 2027 revenue targets of $350 million to $400 million and identified a $500 million cross-selling opportunity across the combined customer base. The deal positions SoundHound to compete more directly with Microsoft, Google, and Amazon across enterprise contact centers, healthcare, and financial services.\n\nSource:Simply Wall St ↗\n\n### (AI News) Nvidia's Memory Costs Soar 485% as Latest AI Systems Now Cost $7.8 Million to Build\n\nDate: May 21, 2026 [...] Source:JDJournal\n\n### (AI News) US Cuts Cyber Patch Window to Three Days as AI Threats Rise\n\nDate: June 12, 2026\n\nSummary: The US Cybersecurit

In [51]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [25]:
# LLM

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0, 
    api_key=OPENAI_API_KEY
)

In [24]:
response = llm.invoke('what year is it?')
response 

AIMessage(content='It is currently 2022.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 12, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Eb2mFXnYzqHUgpPWyk8YJtcajRLfu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1592d-7984-7771-b68d-702ca388eb80-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 7, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [31]:
prompt = Client().pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [ ]:
tools = [search_tool, get_weather_data]

In [66]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
    )

In [70]:
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True
    )

In [68]:
response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...
I should first search for the capital of India and then fetch the current weather information for that city.
Action: tavily_search_results_json
Action Input: "capital of India"[{'title': 'List of cities that served as the capital of India | India News - Times of India', 'url': 'https://timesofindia.indiatimes.com/india/list-of-cities-that-served-as-the-capital-of-india/articleshow/111815680.cms', 'content': "| Era | Capital(s) | Description |\n| Modern | Allahabad | Capital of India for a day in 1858, capital of North-Western Provinces. |\n| Calcutta | Capital of British India until 1911. |\n| Shimla | Summer capital by the latter half of the 19th century. |\n| New Delhi | Capital of India since 1911, buildings inaugurated in early 1931. |\n| Mediaeval | Agra | Capital of Sikandar Lodi during the Lodhi dynasty, moved from Delhi in 1506. |\n| Vijayanagara | Capital of Vijayanagara Empire until 1585. |\n| Patna | Capital of Sher Shah Suri's Empire 

In [69]:
print(response["output"])

The capital of India is New Delhi and the current weather in New Delhi is 27°C with Haze and 34% humidity.
